# B2 - resume of B (after the abstain C3 row hit 256 tokens)
Reuses the 128-token and 256-token generation files from notebook B (stored in the dataset under `resume_inputs/`), regenerates only the still-truncated row at **512** (owner decision 2026-11-08, DEVIATIONS 4b), validates, then runs extraction and labels exactly as in B.

In [ ]:
import os, shutil, glob, json, subprocess
hits = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/FROZEN_v2.json', recursive=True)]
assert hits, 'attach the rag-noise-probe dataset (with artifacts/data/FROZEN_v2.json)'
SRC = os.path.dirname(os.path.dirname(hits[0])); REPO = '/kaggle/working/rag-noise-probe'
shutil.copytree(SRC, REPO, dirs_exist_ok=True); os.chdir(REPO)
!pip install -q rank_bm25 pyarrow pytest 2>&1 | tail -1
!python -m pytest tests -q 2>&1 | tail -2
import transformers, torch; print('transformers', transformers.__version__, 'torch', torch.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
_tok, _label = None, None
for _l in ['HF_TOKEN', 'hf_token', 'HF_token', 'Hf_token', 'HUGGINGFACE_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'huggingface_token', 'HF']:
    try:
        _tok = UserSecretsClient().get_secret(_l); _label = _l; break
    except Exception as e:
        print('secret label', repr(_l), '-> not available', type(e).__name__)
assert _tok, 'No Hugging Face secret reachable from this run (attach it under Add-ons -> Secrets)'
print('using secret label:', repr(_label))   # the value is never printed
login(token=_tok); del _tok
SECRET_ONLY = os.environ.get('SECRET_ONLY') == '1'
MODEL = 'meta-llama/Llama-2-7b-chat-hf'          # revision is pinned in artifacts/data/model_pin.json
BENCH = 'artifacts/data/benchmark_v2.jsonl'
RUN = 'artifacts/runs/llama2-7b'; os.makedirs(RUN, exist_ok=True)
CFG = json.load(open('artifacts/data/experiment_config_v2.json'))
MAX_NEW_TOKENS = CFG['generation']['max_new_tokens']; assert MAX_NEW_TOKENS == 128      # FROZEN 2026-11-07
assert CFG['revision'] == json.load(open('artifacts/data/model_pin.json'))['revision']
print('frozen config: max_new_tokens', MAX_NEW_TOKENS, '| regen', CFG['generation']['regenerate_truncated_with_max_new_tokens'], '| ranking', CFG['head_layer_selection']['primary'])

In [ ]:
!python -m src.gen.model_check --bench $BENCH --out $RUN/model_check.json
assert json.load(open(f'{RUN}/model_check.json'))['all_ok'], 'model check failed - stop'

In [ ]:
import glob, shutil
src = os.path.dirname(glob.glob('/kaggle/input/**/resume_inputs/gen_std_128.jsonl', recursive=True)[0])
for f in glob.glob(src + '/*'):
    shutil.copy(f, RUN)
!ls -la $RUN

## Selective regeneration at 256 for rows without EOS within 128, merge, VALIDATE (stops the notebook if validation fails)

In [ ]:
!python -m src.gen.finalize_gen --model $MODEL --bench $BENCH --template std --run $RUN
!python -m src.gen.finalize_gen --model $MODEL --bench $BENCH --template abstain --run $RUN
for t in ('std', 'abstain'):
    r = json.load(open(f'{RUN}/gen_{t}_final_report.json'))
    print(t, {k: r[k] for k in ('rows', 'truncated_after_first_pass', 'regenerated_rows', 'still_truncated', 'empty_rate', 'merge_stats')})
    assert not r['errors'] and r['rows'] == 750, f'{t}: generation validation failed - stop'

## ReDeEP + probe extraction (std answers) + token dump for the 100 comparison rows

In [ ]:
!python -m src.extract.redeep_extract --model $MODEL --gen $RUN/gen_std.jsonl --bench $BENCH --outdir $RUN --cmp_set artifacts/data/comparison_set_v2.json --checkpoint_every 25

## Labels (+ label-change report vs Day-1 substring rule) and G3 sanity check

In [ ]:
!python -m src.data.evaluator --gen $RUN/gen_std.jsonl --bench $BENCH --out $RUN/labeled_std.jsonl
!python -m src.data.evaluator --gen $RUN/gen_abstain.jsonl --bench $BENCH --out $RUN/labeled_abstain.jsonl
import pandas as pd, numpy as np
lab = pd.DataFrame([json.loads(l) for l in open(f'{RUN}/labeled_std.jsonl')])
f = pd.read_parquet(f'{RUN}/redeep_feats.parquet').merge(lab[['uid','condition','failure']], on='uid')
c1 = f[f.condition == 'C1_gold']; P = np.stack(c1.pks_mean.values); late = P[:, P.shape[1]//2:].mean(1)
print('rows', len(f), '| truncated answers', int((~lab.stopped_by_eos).sum()), '| tf_match', f.tf_match.mean().round(3))
print('G3: late-layer PKS fail=%.2f ok=%.2f' % (late[c1.failure==1].mean(), late[c1.failure==0].mean()))

In [ ]:
import zipfile
with zipfile.ZipFile('/kaggle/working/full_run_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root in ['artifacts/runs/llama2-7b']:
        for dp, dn, fn in os.walk(root):
            if '.parts' in dp: continue
            for n in fn:
                p = os.path.join(dp, n); z.write(p, p.replace(os.sep, '/'))
print(os.path.getsize('/kaggle/working/full_run_outputs.zip') / 1e6, 'MB')